# Praktikum 5: Notebook 2: Failiformaadid ja partitsioneerimine

**Moodul 5: Suurandmed ja pilvelahendused (edasijõudnud)**

**Hinnanguline aeg:** 25 minutit

**Eeldused:** Läbitud notebook 01. Praktikumi [README](./README.md) sammud on tehtud, `samples.nyctaxi.trips` ja Azure Open Datasets on saadaval.

## Mida me õpime

Sellel notebook'il on kaks eesmärki:

1. **Mõõdame**, kui palju kiiremini loetakse Parquet ja Delta võrreldes CSV-ga sama andmehulga peal.
2. **Näeme**, kuidas `partitionBy` muudab päringu teed: `EXPLAIN` plaan tõestab, et Spark loeb ainult vajalikud failid (partition pruning).

Kasutame päris NYC Yellow Taxi andmeid. Seal on miljonid read, mis peaks andma mõõdetavaid erinevusi.

## 1. Valmistame ette kolm varianti

Võtame väikese, aga tähenduslikku osa (ühe kuu) ja kirjutame selle kolmes formaadis:

- **CSV**: tekstipõhine, kõige aeglasem, aga tuttav
- **Parquet**: veerupõhine, kompressitud
- **Delta**: Parquet + transaktsiooni logi

NB! Kirjutame piiratud osa, et mahutada Free Edition kvoodi sisse. 2 miljonit rida on piisav, et erinevus oleks selgelt näha, aga mitte nii palju, et kvoot täis tuleks (võite ka katsetada muidugi suurema mahuga :-)).

In [ ]:
# NYC Yellow Taxi andmed Azure Open Datasets hoidlast
yellow_path = "wasbs://nyctlc@azureopendatastorage.blob.core.windows.net/yellow"
yellow = spark.read.parquet(yellow_path)
print("Ridu kokku:", yellow.count())

# Kasutame aktiivset kataloogi ja skeemi, et kood toimiks igas workspace'is
# (Free Edition'is on vaikimisi `main.default`, aga ei pruugi alati nii olla)
catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
schema  = spark.sql("SELECT current_schema()").collect()[0][0]

# Volume on UC-poolne hallatud failihoidla - siia kirjutame CSV / Parquet / Delta failid
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.praktikum5")
base = f"/Volumes/{catalog}/{schema}/praktikum5"
print("Kirjutame kausta:", base)

In [ ]:
# Võtame fikseeritud suuruse, et võrdlus oleks aus
import pyspark.sql.functions as F

sample = (
    yellow
    .select(
        "tpepPickupDateTime",
        "tpepDropoffDateTime",
        "passengerCount",
        "tripDistance",
        "fareAmount",
        "totalAmount",
        "paymentType",
    )
    .withColumn("pickupYear",  F.year("tpepPickupDateTime"))
    .withColumn("pickupMonth", F.month("tpepPickupDateTime"))
    .filter("pickupYear == 2012")
    .limit(2_000_000)
)

print("Ridu töösse võetud:", sample.count())

In [ ]:
# Kirjutame CSV-sse (aeglane aga võrdluseks kasulik)
sample.write.mode("overwrite").option("header", True).csv(f"{base}/csv")

# Kirjutame Parqueti (ilma partitsioneerimiseta)
sample.write.mode("overwrite").parquet(f"{base}/parquet")

# Kirjutame Delta'sse (ilma partitsioneerimiseta)
sample.write.mode("overwrite").format("delta").save(f"{base}/delta")

# Kirjutame Parquet'i PARTITSIONEERITUNA aasta-kuu järgi
(
    sample.write
    .mode("overwrite")
    .partitionBy("pickupYear", "pickupMonth")
    .parquet(f"{base}/parquet_partitioned")
)

print("Kõik neli varianti salvestatud kausta", base)

## 2. Mõõdame lugemiskiirust

Spark on laisk: `spark.read` iseenesest ei loe midagi. Et lugemiskiirust mõõta, peame tegema action'i, mis sunnib kõigi ridade lugemise. `count()` teeb selle.

NB! Mõõtmine Spark'is ei ole kunagi perfektne (mõjutavad nt vahemälu, optimeerija, külmkäivitus). Tulemused on suurusjärgud, mitte täpsed benchmarkid.

In [ ]:
import time

# NB! Serverless compute'il ei saa vahemälu käsitsi tühjendada (CLEAR CACHE pole lubatud).
# read_fn loob iga kord värske DataFrame'i ja `sample` pole cache'itud, nii et
# võrdlus on ikkagi aus — aga esimesed read'id võivad olla veidi aeglasemad
# külmkäivituse tõttu. Keskendu suurusjärkudele, mitte sekundi täpsusele.

def measure(label, read_fn):
    t0 = time.time()
    rows = read_fn().count()
    dt = time.time() - t0
    print(f"{label:30s}  {rows:>10,d} rida  {dt:>6.2f} s")
    return dt

t_csv = measure(
    "CSV",
    lambda: spark.read.option("header", True).csv(f"{base}/csv"),
)
t_parquet = measure(
    "Parquet",
    lambda: spark.read.parquet(f"{base}/parquet"),
)
t_delta = measure(
    "Delta",
    lambda: spark.read.format("delta").load(f"{base}/delta"),
)

**Miks Parquet on kiirem kui CSV?** Parquet on veerupõhine: `count(*)` ei vaja tegelikult veergude sisu lugemist, piisab metadata'st. CSV puhul peab Spark iga rea algusest lõpuni parsima.

**Miks Delta ja Parquet on umbes sama kiirusega?** Delta on Parquet + transaktsiooni logi. Lugemine läheb sama Parquet failide peale, ainult lisaks vaatab `_delta_log` kataloogi, et teada, millised failid on aktiivsed.


Viide: [Parquet formaat](https://parquet.apache.org/documentation/latest/)

## 3. Partition pruning: reaalne efekt

Teeme sama päringu kaks korda: ühel juhul partitsioneerimata Parquet, teisel juhul partitsioneeritud. Filter on `pickupMonth = 1`.

Ootus: partitsioneeritud variandi puhul loeb Spark ainult ühe partition-kausta, partitsioneerimata variandi puhul kogu andmestiku.


Viide: [Spark andmeallikad ja partitsioneerimine](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#partition-discovery)

In [ ]:
# Partitsioneerimata
t_flat = measure(
    "Parquet (flat) + filter",
    lambda: spark.read.parquet(f"{base}/parquet").where("pickupMonth = 1"),
)

# Partitsioneeritud
t_part = measure(
    "Parquet (partitioned) + filter",
    lambda: spark.read.parquet(f"{base}/parquet_partitioned").where("pickupMonth = 1"),
)

In [ ]:
# EXPLAIN plaan: siin näeme, kas partition pruning tegelikult käivitub
df_flat = spark.read.parquet(f"{base}/parquet").where("pickupMonth = 1")
df_part = spark.read.parquet(f"{base}/parquet_partitioned").where("pickupMonth = 1")

print("=== Partitsioneerimata ===")
df_flat.explain()

print("\n=== Partitsioneeritud ===")
df_part.explain()

Vaata EXPLAIN väljundis kahte rida:

- `PartitionFilters:`: siin ilmub partitsioneeritud variandi puhul `[(pickupMonth#... = 1)]`. See tähendab, et Spark rakendab filtri enne failide lugemist.
- `PushedFilters:`: see on filter, mille Spark surub faili sisse (row-level). Mõlemas variandis on see olemas.

`PartitionFilters` on võti. Kui see on olemas ja sul on õigesti partitsioneeritud andmed, loeb Spark ainult õiged failid.

## 4. Vaatame failisüsteemi

Partitsioneerimine Sparkis tähendab, et iga partition saab eraldi kausta. Vaatame, kuidas see kettal välja näeb.

In [ ]:
# Juurkaust näitab aasta-kuu partitsioone
for f in dbutils.fs.ls(f"{base}/parquet_partitioned"):
    print(f.path, f.size)

In [ ]:
# Üks konkreetne partition: siin on tegelik parquet fail
# Leiame dünaamiliselt esimese aasta- ja kuu-partitsiooni
years = [p for p in dbutils.fs.ls(f"{base}/parquet_partitioned") if p.name.startswith("pickupYear=")]
months = [p for p in dbutils.fs.ls(years[0].path) if p.name.startswith("pickupMonth=")]
print(f"Näitame partition-kausta: {months[0].path}\n")

for f in dbutils.fs.ls(months[0].path):
    print(f.path, f.size)

Kausta nimi kujul `pickupYear=2012` on Hive-style partition encoding. Spark (ja teised tööriistad nagu Presto, Athena, BigQuery external) oskavad seda lugeda ilma metaandmeteta. Selle saab ka tabeli kujul Unity Catalogi registreerida. Siis `SELECT * FROM main.default.trips WHERE pickupMonth = 1` kasutab sama pruning loogikat.

## 5. Partitsioneerimise valiku arhitektuurne põhjendus

Partitsioneerimise valik pole tehniline triviaalsus, vaid arhitektuuriline otsus.

**Probleem.** Päring loeb suurest andmestikust ainult väikese osa, aga peab läbima kõik failid. See on aeglane ja kallis.

**Variandid.**

1. Ei partitsioneeri: lihtsam, aga iga päring loeb kõike.
2. Partitsioneerime kuu järgi (`year`, `month`): sobib ajalistele päringutele, enamik analüütikast loeb viimast kuud või kvartalit.
3. Partitsioneerime kasutaja/kliendi ID järgi: sobib, kui enamik päringuid filtreerib kliendi järgi, aga teeb failistruktuuri "väikeste failide põrguks", kui kliente on palju.

**Valik ja põhjendus.** NYC Taxi puhul on aja järgi partitsioneerimine selge võit: kõige sagedasem päringumuster on "mis toimus perioodil X kuni Y". `year`-`month` on hea granulaarsus. Mitte liiga suur kausta kohta (GB-d on OK), mitte liiga palju partitsioone.

**Kompromissid.** Kui keegi teeb päringu *üle kogu ajaloo* (nt "leia kõige pikemad sõidud kunagi"), peab Spark ikka kõiki faile lugema. Partition pruning ei aita. Samuti, kui lisad andmeid vanale kuule, peab Spark terve partitsiooni üle kirjutama. Otsus vaadatakse üle siis, kui päringumuster muutub (näiteks analüütikud hakkavad filtreerima kliendi ID järgi).

## 6. Kokkuvõttev võrdlus

| Formaat | Skeem | ACID | Lugemise kiirus (count) | Kasutus |
|---------|-------|------|-------------------------|---------|
| CSV | Käsitsi defineeritud või inferitud iga kord | Ei | Aeglane (rida-rida parse) | Vahetus, import/export |
| Parquet | Failis kaasas | Ei | Kiire (metadata count) | Analüütika, data lake |
| Delta | Failis kaasas + `_delta_log` | Jah | Kiire + versioonimine | Lakehouse, Production pipeline'id |

Partitsioneerimine on ortogonaalne formaadivalikule. Saad partitsioneerida Parquet, Delta, ORC. Delta eelis on see, et lisaks toetab ta `OPTIMIZE` ja `Z-ORDER` operatsioone, mis teevad _pruning_u veel efektiivsemaks.


Viide: [Delta Lake dokumentatsioon](https://docs.delta.io/latest/index.html)

### Proovi ise

Tee järgmist:

1. Kirjuta `sample` veel ühes variandis kettale: partitsioneeritud **ainult `paymentType`** järgi (mitte aja järgi).
2. Mõõda sama päring `paymentType = 1` filtri peal mõlema variandi (aja-partitsioneeritud ja maksetüübi-partitsioneeritud) peal.
3. Vaata, kumb on kiirem ja vaata `EXPLAIN`. Põhjenda, miks päringut filtreerides `paymentType = 1` on maksetüübi-partitsioneering efektiivsem, aga aja-päringute puhul on olukord vastupidi.

See harjutus illustreerib peamist partitsioneerimise põhimõtet: **õige partitsioneering sõltub sellest, kuidas sa andmeid pärid, mitte sellest, kuidas sa neid salvestad.**

In [ ]:
# Sinu lahendus


## Kokkuvõte

Selles notebook'is sa:

- Kirjutasid sama andmestiku CSV, Parquet, Delta ja partitsioneeritud Parquet formaatides
- Mõõtsid lugemiskiirusi ja nägid, et Parquet on CSV-st oluliselt kiirem
- Kasutasid `EXPLAIN` plaani, et tõestada partition pruning töötab
- Uurisid partition-kausta struktuuri Hive-stiilis
- Põhjendasid partitsioneerimise valikut arhitektuurses raamistikus

**Mida meeles pidada:**

- Formaadivalik (CSV vs Parquet/Delta) mõjutab juba lugemiskiirust suurusjärkudes.
- Partitsioneerimine aitab ainult siis, kui päring filtreerib partition-võtme järgi. `EXPLAIN` `PartitionFilters` rea olemasolu on lakmustest.
- Hea partition-võti on valik, mis rahuldab enamiku reaalseid päringuid ja hoiab partitsioonide arvu mõistlikuna (ei liiga suur, ei liiga väike).

Järgmine notebook (03_delta_lake_lakehouse.ipynb) läheb edasi ja näitab, mida Delta Lake annab võrreldes tavalise Parquet'iga. ACID, time travel ja medaljoni arhitektuur.